# Qwen3Loop-0.6B: Full-Parameter Tool-Calling SFT
### Fine-Tuning Supervisionado Recorrente com Multi-Exit Loss e AdamW 8-bit

> **Status:** Treinamento supervisionado dos 594M parâmetros físicos com pesos amarrados (tied loop), unrolling 56 camadas lógicas e garantia de sobrevivência à quantização GGUF Q8_0.

In [ ]:
# 1. Verificar GPU Tesla T4 ativa
!nvidia-smi

In [ ]:
# 2. Instalar dependências necessárias
!pip uninstall -y torchao
!pip install -q transformers datasets accelerate bitsandbytes

In [ ]:
# 3. Mapear e preparar diretório de trabalho
import os, sys, shutil, tarfile

print('Configurando ambiente no Kaggle...')

# Descompactar arquivos .tar se presentes em /kaggle/input
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.tar'):
            tar_p = os.path.join(root, f)
            print('Descompactando tar:', tar_p)
            try:
                with tarfile.open(tar_p) as t:
                    t.extractall('/kaggle/working')
            except Exception as e:
                print('Aviso ao extrair:', e)

# 1. Localizar engine qwen3loop_python: DETERMINISTICO (sft-data, com marker)
candidatos = []
for base in ['/kaggle/working', '/kaggle/input']:
    for root, dirs, files in os.walk(base):
        if 'modeling_qwen3loop.py' in files and 'configuration_qwen3loop.py' in files:
            candidatos.append(root)
print('candidatos engine:', candidatos)
def tem_marker(c):
    mk = os.path.join(c, 'modeling_qwen3loop.py')
    try:
        return 'filtered_kwargs' in open(mk).read()
    except Exception:
        return False
marcados = [c for c in candidatos if tem_marker(c)]
print('com marker:', marcados)
sft = [c for c in marcados if 'sft-data' in c]
engine_src = sorted(sft)[0] if sft else None
if engine_src is None:
    work = [c for c in marcados if '/kaggle/working' in c]
    engine_src = sorted(work)[0] if work else None
assert engine_src is not None, 'Nenhuma engine COM marker encontrada (nem sft-data nem tar extraido)!'

engine_src = None
search_paths = ['/kaggle/working', '/kaggle/input']
for base in search_paths:
    for root, dirs, files in os.walk(base):
        if 'modeling_qwen3loop.py' in files and 'configuration_qwen3loop.py' in files:
            if 'sft-data' in root or '/kaggle/working' in root:
                engine_src = root
                break
            elif engine_src is None:
                engine_src = root
    if engine_src and ('/kaggle/working' in engine_src or 'sft-data' in engine_src):
        break

print('engine_src selecionada:', engine_src)

# Copiar engine para /kaggle/working/qwen3loop_python e /kaggle/working/engine/qwen3loop_python
os.makedirs('/kaggle/working/engine', exist_ok=True)
if engine_src != '/kaggle/working/qwen3loop_python':
    shutil.copytree(engine_src, '/kaggle/working/qwen3loop_python', dirs_exist_ok=True)
if engine_src != '/kaggle/working/engine/qwen3loop_python':
    shutil.copytree(engine_src, '/kaggle/working/engine/qwen3loop_python', dirs_exist_ok=True)

# Inserir no início absoluto de sys.path e PYTHONPATH
sys.path.insert(0, '/kaggle/working')
sys.path.insert(0, '/kaggle/working/engine')
os.environ['PYTHONPATH'] = f'/kaggle/working:/kaggle/working/engine:{os.environ.get("PYTHONPATH", "")}'

# Testar importação e registro formal da arquitetura
import qwen3loop_python
from qwen3loop_python import Qwen3LoopConfig, Qwen3LoopForCausalLM
from transformers import AutoConfig, AutoModelForCausalLM
AutoConfig.register('qwen3loop', Qwen3LoopConfig)
AutoModelForCausalLM.register(Qwen3LoopConfig, Qwen3LoopForCausalLM)
print('Engine qwen3loop_python importada e registrada com sucesso de:', getattr(qwen3loop_python, '__file__', 'unknown'))

# 2. Localizar Modelo Base e Dados
model_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'model.safetensors' in files and 'config.json' in files:
        model_path = root
        break

data_path = None
for base in ['/kaggle/working', '/kaggle/input']:
    for root, dirs, files in os.walk(base):
        if 'curriculum_3000_tool_sft.jsonl' in files:
            data_path = os.path.join(root, 'curriculum_3000_tool_sft.jsonl')
            break
    if data_path:
        break

script_path = None
for base in ['/kaggle/working', '/kaggle/input']:
    for root, dirs, files in os.walk(base):
        if 'train_sft_tool_qwen3loop.py' in files:
            script_path = os.path.join(root, 'train_sft_tool_qwen3loop.py')
            break
    if script_path:
        break

print('Modelo Base detectado:', model_path)
print('Dataset SFT detectado:', data_path)
print('Script SFT detectado: ', script_path)
assert model_path is not None, 'Modelo não encontrado!'
assert data_path is not None, 'Dataset não encontrado!'
assert script_path is not None, 'Script de treino não encontrado!'


In [ ]:
# 4. Executar Treinamento Full-Parameter SFT em Dual GPU Tesla T4 (Accelerate DDP)
output_dir = '/kaggle/working/qwen3loop_tool_curriculum_3000_final'
dev_path = os.path.join(os.path.dirname(data_path), 'dev_hard_v31_60.jsonl')
mask_path = os.path.join(os.path.dirname(data_path), 'mask_first_turn_68.json')

# eval periodica off (treino curto, sem early stopping); val_frac 0 = 100% treino
!export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True && accelerate launch \
    --multi_gpu \
    --num_processes 2 \
    --num_machines 1 \
    --mixed_precision no \
    --dynamo_backend no \
    {script_path} \
    --model_path {model_path} \
    --data_path {data_path} \
    --output_dir {output_dir} \
    --batch_size 1 \
    --grad_accum 8 \
    --epochs 2 \
    --lr 1.5e-5 \
    --max_len 5000 \
    --log_steps 1 \
    --eval_steps 1000000 \
    --val_frac 0.0 \
    --mask_first_turn_file {mask_path} \
    --patience 1 \
    --dev_eval {dev_path} \
    --save_steps 35


In [ ]:
import os
final = '/kaggle/working/qwen3loop_tool_curriculum_3000_final/model.safetensors'
assert os.path.exists(final) and os.path.getsize(final) > 10**9, 'TREINO NAO GEROU PESOS (model.safetensors ausente/pequeno)!'
print('pesos OK:', os.path.getsize(final))
# 5. Compactar os pesos finais do modelo para download direto
!cd /kaggle/working && zip -r qwen3loop_tool_curriculum_3000_final.zip qwen3loop_tool_curriculum_3000_final/
!ls -lh /kaggle/working/qwen3loop_tool_curriculum_3000_final.zip
print('\nTreinamento SFT concluído com sucesso! Checkpoint zip gerado para download e unrolling.')